# Spaceship Titanic - Solucion de Competicion

**Competicion Kaggle**: [Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic)

**Mejor Puntuacion**: 80.90% (Leaderboard Publico puesto 206 de 2684 participantes)

## Resumen del Enfoque

Este notebook documenta el proceso para alcanzar aproximadamente 81% de precision en la competicion Spaceship Titanic:

1. **Ingenieria de Caracteristicas** - Imputacion basada en grupos, caracteristicas de cabina, patrones de gasto
2. **Seleccion de Modelo** - Se probaron LightGBM, XGBoost, CatBoost y ensambles
3. **Regularizacion** - Regularizacion fuerte para reducir la brecha entre entrenamiento y test
4. **Promediado de Semillas** - Multiples semillas aleatorias para reducir varianza

**Hallazgo Clave**: CatBoost con promediado de semillas tuvo el mejor rendimiento, generalizando bien desde CV al leaderboard publico.

## 1. Configuracion y Carga de Datos

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# Cargar datos
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f"Forma del train: {train.shape}")
print(f"Forma del test: {test.shape}")
train.head()

## 2. Exploracion de Datos

In [ ]:
# Distribucion del objetivo
print("Distribucion del Objetivo:")
print(train['Transported'].value_counts(normalize=True))

# Valores faltantes
print("\nValores Faltantes:")
print(train.isnull().sum())

## 3. Ingenieria de Caracteristicas

Caracteristicas principales extraidas:
- **Caracteristicas de grupo**: Del PassengerId (grupos viajando juntos)
- **Caracteristicas de cabina**: Deck, CabinNum, Side de la columna Cabin
- **Caracteristicas de gasto**: Gasto total, transformaciones logaritmicas, patrones de gasto
- **Caracteristicas derivadas**: IsChild, AgeBin, CabinNumEven

In [ ]:
# Guardar IDs para la submission
test_ids = test['PassengerId'].copy()

# Objetivo
y = train["Transported"].astype(int).values

# Combinar train y test para procesamiento consistente
train["is_train"] = 1
test["is_train"] = 0
full = pd.concat([train, test], ignore_index=True)

print(f"Forma de datos combinados: {full.shape}")

In [ ]:
# === CARACTERISTICAS DE GRUPO ===
# Formato PassengerId: GGGG_PP (Grupo_PersonaEnGrupo)
pid_split = full["PassengerId"].str.split("_", expand=True)
full["Group"] = pid_split[0].astype(int)
full["GroupSize"] = full.groupby("Group")["PassengerId"].transform("count")
full["IsAlone"] = (full["GroupSize"] == 1).astype(int)

print("Distribucion de Tamano de Grupo:")
print(full["GroupSize"].value_counts().head(10))

In [ ]:
# === CARACTERISTICAS DE CABINA ===
# Formato Cabin: Deck/Num/Side (ej., B/0/P)
cabin_split = full["Cabin"].str.split("/", expand=True)
full["Deck"] = cabin_split[0]
full["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
full["Side"] = cabin_split[2]

print("Distribucion de Deck:")
print(full["Deck"].value_counts())

In [ ]:
# === CARACTERISTICAS DE GASTO ===
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
full["TotalSpend"] = full[spend_cols].sum(axis=1)
full["HasSpend"] = (full["TotalSpend"] > 0).astype(int)
full["NumAmenitiesUsed"] = full[spend_cols].gt(0).sum(axis=1)

print("Estadisticas de Gasto:")
print(full["TotalSpend"].describe())

## 4. Imputacion de Valores Faltantes

**Estrategia Clave**: Imputacion basada en grupos
- Las personas en el mismo grupo frecuentemente comparten caracteristicas (HomePlanet, Destination, etc.)
- Usar moda del grupo para categoricas, mediana del grupo para numericas

In [ ]:
# Imputacion basada en grupo para columnas categoricas
group_mode_cols = ["HomePlanet", "Destination", "Deck", "Side", "CryoSleep", "VIP"]
for col in group_mode_cols:
    full[col] = full.groupby("Group")[col].transform(
        lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
    )

# Imputacion basada en grupo para columnas numericas
for col in ["Age", "CabinNum"]:
    full[col] = full.groupby("Group")[col].transform(lambda x: x.fillna(x.median()))
    full[col] = full[col].fillna(full[col].median())  # Respaldo con mediana global

# Rellenar gastos con 0 (sin gasto)
for col in spend_cols:
    full[col] = full[col].fillna(0)

# Logica de CryoSleep: si esta en criosueño, no puede gastar dinero
cs_na = full["CryoSleep"].isna()
full.loc[cs_na & (full["TotalSpend"] == 0), "CryoSleep"] = True
full.loc[cs_na & (full["TotalSpend"] > 0), "CryoSleep"] = False
full["CryoSleep"] = full["CryoSleep"].fillna(False)
full["VIP"] = full["VIP"].fillna(False)

# Categoricas restantes con moda
for col in ["HomePlanet", "Destination", "Deck", "Side"]:
    full[col] = full[col].fillna(full[col].mode()[0])

print("Valores faltantes despues de imputacion:")
print(full.isnull().sum().sum())

## 5. Caracteristicas Derivadas

In [ ]:
# Caracteristicas basadas en edad
full["IsChild"] = (full["Age"] < 18).astype(int)
full["AgeBin"] = pd.cut(full["Age"], bins=[-1, 12, 18, 25, 40, 60, 200], labels=False).astype(int)

# Caracteristica de posicion de cabina (patron par/impar observado en datos)
full["CabinNumEven"] = (full["CabinNum"] % 2 == 0).astype(int)

# Transformaciones logaritmicas para gastos (reduce asimetria)
full["LogTotalSpend"] = np.log1p(full["TotalSpend"])
for col in spend_cols:
    full["Log_" + col] = np.log1p(full[col])

print("Caracteristicas creadas exitosamente!")

## 6. Codificacion y Preparacion de Datos

In [ ]:
# Eliminar columnas no necesarias para modelado
full_features = full.drop(columns=["Cabin", "Name", "Group"])

# Codificar columnas categoricas
cat_cols = full_features.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    full_features[col] = LabelEncoder().fit_transform(full_features[col].astype(str))

# Convertir booleanos a int
bool_cols = full_features.select_dtypes(include="bool").columns.tolist()
for col in bool_cols:
    full_features[col] = full_features[col].astype(int)

# Preparar columnas de caracteristicas
feature_cols = [c for c in full_features.columns if c not in ["Transported", "is_train", "PassengerId"]]

# Separar de nuevo en train y test
train_processed = full_features[full_features["is_train"] == 1].copy()
test_processed = full_features[full_features["is_train"] == 0].copy()

X = train_processed[feature_cols].astype(float)
X_test = test_processed[feature_cols].astype(float)

print(f"Numero de caracteristicas: {len(feature_cols)}")
print(f"Lista de caracteristicas: {feature_cols}")

## 7. Entrenamiento del Modelo - CatBoost con Promediado de Semillas

**Por que CatBoost?**
- Mejor generalizacion entre los modelos probados (LightGBM, XGBoost, CatBoost)
- Menor brecha entre puntuacion CV y puntuacion del leaderboard publico

**Por que Promediado de Semillas?**
- Reduce la varianza en las predicciones
- Cada semilla produce modelos ligeramente diferentes
- El promediado suaviza las fluctuaciones aleatorias

In [ ]:
# Multiples semillas para promediado
seeds = [42, 123, 456, 789, 2024, 1337, 7777, 8888, 9999, 13]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_test_proba = []
all_oof_proba = []

print(f"Entrenando con {len(seeds)} semillas diferentes...")
print("="*50)

for seed in seeds:
    oof_proba = np.zeros(len(X))
    test_proba = np.zeros(len(X_test))
    
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y), 1):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]
        
        model = CatBoostClassifier(
            n_estimators=500,
            learning_rate=0.03,
            depth=5,
            subsample=0.6,
            l2_leaf_reg=3.0,
            min_data_in_leaf=40,
            random_state=seed,
            verbose=0
        )
        model.fit(X_tr, y_tr)
        oof_proba[val_idx] = model.predict_proba(X_val)[:, 1]
        test_proba += model.predict_proba(X_test)[:, 1] / cv.n_splits
    
    cv_acc = accuracy_score(y, (oof_proba >= 0.5).astype(int))
    print(f"Semilla {seed}: CV = {cv_acc:.4f} ({cv_acc*100:.2f}%)")
    
    all_oof_proba.append(oof_proba)
    all_test_proba.append(test_proba)

print("="*50)

In [ ]:
# Promediar predicciones de todas las semillas
avg_oof = np.mean(all_oof_proba, axis=0)
avg_test = np.mean(all_test_proba, axis=0)

# Calcular puntuacion CV promediada
avg_cv = accuracy_score(y, (avg_oof >= 0.5).astype(int))
print(f"\nPuntuacion CV Promediada: {avg_cv:.4f} ({avg_cv*100:.2f}%)")

## 8. Resumen de Resultados

| Enfoque | Puntuacion CV | Kaggle LB |
|---------|---------------|----------|
| LightGBM (baseline) | 81.02% | 80.24% |
| XGBoost | 81.04% | - |
| CatBoost (semilla unica) | 81.28% | 80.83% |
| CatBoost (5 semillas prom.) | 81.38% | 80.90% |
| CatBoost (10 semillas prom.) | 81.26% | aprox. 81% |

**Observaciones Clave**:
- CatBoost generaliza mejor que LightGBM/XGBoost
- El promediado de semillas reduce varianza y mejora la puntuacion LB
- La regularizacion fuerte es crucial para evitar sobreajuste

## 9. Crear Submission

In [ ]:
# Crear submission con threshold 0.5
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": (avg_test >= 0.5).astype(bool)
})

submission.to_csv("submission.csv", index=False)
print(f"Submission guardada con {len(submission)} predicciones")
submission.head()

In [ ]:
# Verificar distribucion de predicciones
print("Distribucion de Predicciones:")
print(submission["Transported"].value_counts(normalize=True))

## 10. Lecciones Aprendidas

1. **Imputacion basada en grupos** es crucial - los pasajeros viajando juntos comparten caracteristicas
2. **CryoSleep** es un predictor fuerte - las personas en criosueño no pueden gastar dinero
3. **La regularizacion importa** - prevenir sobreajuste con restricciones apropiadas
4. **Promediado de semillas** - tecnica simple que mejora consistentemente las puntuaciones
5. **Seleccion de modelo** - CatBoost generalizo mejor para este dataset

---
*Notebook creado para la Competicion Spaceship Titanic de Kaggle*